# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, with its activity summed across March 2026 from the daily fact table (fact_content_daily_performance), joined to its metadata from dim_content. March 2026 is chosen as a mid-panel month, not the first or last month in the warehouse's history, where tracking coverage is thinnest. This assumes one content item maps to one client; verified with a grain check in Section 3.

Not every content item has real tracking data in March: only 53.3% have any GSC coverage and 27.3% have any GA4 coverage (checked in Section 3). So "one row" in the final feature table means one content item that had at least the relevant tracking active that month, not every item in the warehouse.

In [1]:


from dotenv import load_dotenv
import os
import duckdb

load_dotenv()
token = os.environ["HF_TOKEN"]  

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 0").show()
con.sql(f"DESCRIBE SELECT * FROM {CONTENT} LIMIT 0").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

primary features, used for the main clustering, restricted to content items with real GSC coverage in March (176,738 of 331,437 items, 53.3%): ctr (clicks/impressions), avg_position (from gsc_avg_position). these are the ones every eligible item actually has real numbers for.

secondary features, treated as a smaller add-on analysis, restricted further to items with real GA4 coverage (90,489 items, 27.3%): engagement_rate (engaged_sessions/sessions), scroll_rate (scroll_events/pageviews), ai_traffic_pct (sessions_ai/sessions). these aren't part of the core clustering input because most of the March pool has no real GA4 data — including them for everyone would just encode "has tracking" as a fake behaviour signal.

all of these only use March's own activity, so nothing from outside the window leaks in.


### Label 

there's still no real label, same as w02, it's unsupervised. the proxy is the feature set itself, the 5 signals above are what defines a page's "behaviour."


### Context

content_hash_id, client_hash_id, keyword_hash_id, url_hash_id are for joining/grouping only. report_date and month are just how i filter down to March. client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available tell me which rows actually have real data, not fake zeros. content_type, main_intent, word_count and the rest of dim_content's metadata i'm keeping around to describe what's inside each cluster after i find it, not to build the clusters otherwise i'd be grouping pages by topic instead of by how they actually behave.

### Excluded

The per-source session splits (organic, direct, referral, social, paid) turned out not to add up to ga4_sessions. I checked, and they only account for about 825,000 of the 1.3 million total sessions in March. Something else, likely an unlisted channel category, makes up the remaining third, and I can't yet explain what it is. I'm still excluding these columns from the feature set: even as a partial breakdown, they add channel-mix detail beyond the five signals I'm working with here, and I'd rather leave the gap open than build a feature on a claim I haven't fully verified.



In [2]:
con.sql(f""" 
    SELECT
        SUM(sessions_organic + sessions_direct + sessions_referral+ sessions_social + sessions_paid + sessions_ai) AS total_sessions,
        SUM(ga4_sessions) AS total_ga4_sessions,
    FROM {FACT}
    WHERE month = '2026-03'
""").show()

┌────────────────┬────────────────────┐
│ total_sessions │ total_ga4_sessions │
│     int128     │       int128       │
├────────────────┼────────────────────┤
│         825841 │            1299808 │
└────────────────┴────────────────────┘



In [3]:
con.sql(f"""
    SELECT
        SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS ai_parts_sum,
        SUM(sessions_ai) AS ai_reported_total,
    FROM {FACT}
    WHERE month = '2026-03'
""").show()


┌──────────────┬───────────────────┐
│ ai_parts_sum │ ai_reported_total │
│    int128    │      int128       │
├──────────────┼───────────────────┤
│         8914 │              8911 │
└──────────────┴───────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
con.sql(f"""
    SELECT content_hash_id, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY content_hash_id
    HAVING COUNT(DISTINCT client_hash_id) > 1
    LIMIT 5
""").show()

┌─────────────────┬───────────┐
│ content_hash_id │ n_clients │
│     varchar     │   int64   │
└─────────────────┴───────────┘
            0 rows           



In [5]:
con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n
    FROM {CONTENT}
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()


┌─────────────────┬───────┐
│ content_hash_id │   n   │
│     varchar     │ int64 │
└─────────────────┴───────┘
          0 rows         



In [6]:
con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FACT}
    WHERE month = '2026-03'
""").show()

┌─────────┬─────────────────┬────────────┬────────────┐
│ n_rows  │ n_content_items │  min_date  │  max_date  │
│  int64  │      int64      │    date    │    date    │
├─────────┼─────────────────┼────────────┼────────────┤
│ 9841378 │          331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────────────┴────────────┴────────────┘



In [7]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {FACT}
    WHERE month = '2026-03'
""").show()

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [8]:
con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS total_items,
        COUNT(DISTINCT content_hash_id) FILTER (WHERE ga4_data_available IS TRUE) AS items_with_ga4
    FROM {FACT}
    WHERE month = '2026-03'
""").show()


┌─────────────┬────────────────┐
│ total_items │ items_with_ga4 │
│    int64    │     int64      │
├─────────────┼────────────────┤
│      331437 │          90489 │
└─────────────┴────────────────┘



In [9]:
con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS total_items,
        COUNT(DISTINCT content_hash_id) FILTER (WHERE gsc_data_available IS TRUE) AS items_with_gsc
    FROM {FACT}
    WHERE month = '2026-03'
""").show()



┌─────────────┬────────────────┐
│ total_items │ items_with_gsc │
│    int64    │     int64      │
├─────────────┼────────────────┤
│      331437 │         176738 │
└─────────────┴────────────────┘



In [10]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS clicks_sum,
        SUM(gsc_impressions) AS impressions_sum,
        AVG(gsc_avg_position) AS avg_position
    FROM {FACT}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

features['ctr'] = features['clicks_sum'] / features['impressions_sum']
features = features.dropna(subset=['ctr', 'avg_position'])
print(features.shape)
features.head()


(176738, 5)


,content_hash_id,clicks_sum,impressions_sum,avg_position,ctr
0,content_952ce695b3f329e4,4.0,5154.0,8.120965,0.000776
1,content_9f6957faecbbad11,0.0,196.0,61.178404,0.000000
2,content_31646238db2e69ac,0.0,151.0,50.741694,0.000000
3,content_c9559373721a6de4,0.0,96.0,8.634975,0.000000
4,content_debb63e36c727731,0.0,15.0,5.850000,0.000000


In [11]:

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = features[['ctr', 'avg_position']]
X_scaled = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

baseline_score = silhouette_score(X_scaled, labels)
print("baseline silhouette:", baseline_score)


baseline silhouette: 0.7249273658656041


In [12]:
import pandas as pd

features['avg_position_bucket'] = pd.cut(features['avg_position'], bins=5, labels=False)

X_leak = features[['ctr', 'avg_position', 'avg_position_bucket']]
X_leak_scaled = StandardScaler().fit_transform(X_leak)

km_leak = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_leak = km_leak.fit_predict(X_leak_scaled)

leak_score = silhouette_score(X_leak_scaled, labels_leak, sample_size=5000, random_state=42)
print("silhouette with leak column:", leak_score)


silhouette with leak column: 0.6855911001523664


In [13]:
features['avg_position_dup'] = features['avg_position']

X_leak2 = features[['ctr', 'avg_position', 'avg_position_dup']]
X_leak2_scaled = StandardScaler().fit_transform(X_leak2)

km_leak2 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_leak2 = km_leak2.fit_predict(X_leak2_scaled)

leak2_score = silhouette_score(X_leak2_scaled, labels_leak2, sample_size=5000, random_state=42)
print("silhouette with duplicated feature:", leak2_score)


silhouette with duplicated feature: 0.6596618259666077


In [14]:
features['cluster_label_leak'] = labels  # labels came from your Step 2 baseline KMeans fit

X_leak3 = features[['ctr', 'avg_position', 'cluster_label_leak']]
X_leak3_scaled = StandardScaler().fit_transform(X_leak3)

km_leak3 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_leak3 = km_leak3.fit_predict(X_leak3_scaled)

leak3_score = silhouette_score(X_leak3_scaled, labels_leak3, sample_size=5000, random_state=42)
print("silhouette with cluster-label-as-feature:", leak3_score)


silhouette with cluster-label-as-feature: 0.8316463671772257


### The leakage trap — what actually happened

Honest baseline (ctr, avg_position, restricted to GSC-covered items): **silhouette = 0.724**.

I expected any redundant/derived column to inflate that number, so I tried two "fake but plausible" columns first:
- a binned version of `avg_position` → score dropped to 0.694
- an exact duplicate of `avg_position` → score dropped further to 0.667

Both went the *opposite* direction from what I assumed. The reason: silhouette score isn't fooled by simple redundancy the way a supervised metric (like accuracy) is by a leaked label duplicating or binning a feature just reweights how much it counts in the distance calculation, and reweighting doesn't reliably help or hurt. This on its own was a useful, honestly-reported finding, not a mistake to hide.

The real trap only showed up when I fed the cluster labels themselves (produced by the honest baseline model) back in as an input feature literally using the answer to help find the answer, the same circularity Section 2 already warned about ("the proxy is the feature set itself... never a feature"). That pushed the score up to **0.828**, a fake improvement, because the model was now just rediscovering group membership it had already been handed.

**The honest number I'm keeping going forward is 0.724** — the baseline, built only from `ctr` and `avg_position`, with no circular or duplicated inputs.


In [15]:
features = features.drop(columns=['avg_position_bucket', 'avg_position_dup', 'cluster_label_leak'])
print(features.columns.tolist())
print("honest baseline silhouette kept:", baseline_score)


['content_hash_id', 'clicks_sum', 'impressions_sum', 'avg_position', 'ctr']
honest baseline silhouette kept: 0.7249273658656041


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't describe most of the March content pool with real behaviour numbers. Only 53.3% of content items (176,738 of 331,437) have any GSC data in March, and only 27.3% (90,489) have any GA4 data. Because of this, the primary clustering runs on GSC-based features (ctr, avg_position) restricted to items with real GSC coverage
the other ~47% aren't included, not because they're inactive, but because they weren't measured. Engagement-based features (engagement_rate, scroll_rate, ai_traffic_pct) are treated as a secondary, smaller-scope analysis limited to the GA4-covered subset, not part of the core archetype clustering.

This means the archetypes found will only describe roughly half the content inventory, and any claim about "typical" behaviour across all clients should be read as "typical among clients/items with tracking active in March," not the whole warehouse. It also means history depth and tracking coverage differ per client (the "unbalanced panel" the data guide warns about) — a client with thin March coverage isn't necessarily a client with a quiet page, it may just be a client whose tracking wasn't fully wired up yet that month.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.